# Local video → clips → captions (end-to-end, runs on your Mac)

Take **one local video** (e.g. a 30-min aquarium recording), run the full octopus-clip
**extraction** pipeline on it, then **caption** each extracted clip with the local
**Qwen3-VL-2B MLX 4-bit** caption student — no server, no Colab, no GPU/CUDA.

**Pipeline (identical to the production one, just local + local captioner):**
1. **Extract** — two 1 fps passes over the video: octopus detection (CLIP ViT-B/32 + `clip_mlp_hardneg_v2`,
   letterbox) and absolute motion (`scan_motion_area`). Slide a non-overlapping 20 s window; keep a window
   when **>50 % of frames are octopus-visible (p≥0.6) AND mean motion ≥ 0.008**. ffmpeg byte-range copy.
2. **Caption** — for each clip, prepare frames exactly as the student was trained (dense 1 fps → score with
   the detector → top-6 by p_visible, chronological → thumbnail 768 → CLAHE) and generate one sentence with
   the MLX student. Clips with no confident octopus frame are auto-labelled `octopus not present`.

**To run:** set `VIDEO_PATH` in the first cell, then Run All. Uses `venv/bin/python3` (kernel `octopus-venv`).
Prereq: the MLX model must exist (`models/qwen3vl2b_caption_v1_mlx_4bit/` — build it once via the merge+convert recipe).

## 1. Config — set your video here

In [ ]:
from pathlib import Path

# ============================ SET THIS ============================
VIDEO_PATH = "/path/to/your/30min_video.mp4"   # <-- the local video to process
CAMERA     = "Right_Front"                       # label used only in output filenames (any string)
# =================================================================

# locate repo root (walk up until we find src/)
REPO = Path.cwd()
while not (REPO / "src").exists() and REPO != REPO.parent:
    REPO = REPO.parent
SRC = REPO / "src"

OUT_DIR = REPO / "local_pipeline_out"            # clips + captions land here
# find the MLX caption student (bundled in src/ if packaged, else repo models/)
_cands = [SRC / "qwen3vl2b_caption_v1_mlx_4bit", REPO / "models" / "qwen3vl2b_caption_v1_mlx_4bit"]
MLX_MODEL = next((p for p in _cands if p.exists()), _cands[-1])

# extraction params (defaults copied from src/extract_octopus_clips.py — change to experiment)
SAMPLE_FPS, CLIP_LEN         = 1.0, 20
MIN_VISIBLE_FRAC, VIS_THRESH = 0.50, 0.60
MOTION_THRESH, MOTION_PIX    = 0.008, 25
SIZE, BATCH                  = 224, 64

assert Path(VIDEO_PATH).exists(), f"video not found: {VIDEO_PATH}"
assert MLX_MODEL.exists(), f"MLX model not found — build it first: {MLX_MODEL}"
print("repo :", REPO)
print("video:", VIDEO_PATH)
print("mlx  :", MLX_MODEL)
print("out  :", OUT_DIR)

## 2. Load models — octopus detector (CLIP+MLP) + MLX caption student

In [ ]:
import sys, subprocess, json, tempfile, datetime, time
sys.path.insert(0, str(SRC))
import numpy as np, torch
from PIL import Image

from motion_detector import scan_motion_area                       # absolute per-second motion
from caption_openrouter import (load_detector, extract_frames, score, enhance,
                                 letterbox, N_KEEP, IMG_MAXSIDE, PRESENT_MIN)  # training-identical frame prep
from mlx_vlm import load as mlx_load, generate as mlx_generate
from mlx_vlm.prompt_utils import apply_chat_template

# octopus detector (CLIP ViT-B/32 + MLP probe) — shared by BOTH extraction and caption frame-selection
cm, pre, clf, VIS_IDX, DEV = load_detector()
print("detector loaded on", DEV)

# caption student — the local MLX 4-bit model
mlx_model, mlx_proc = mlx_load(str(MLX_MODEL))
CAP_PROMPT = ("These frames are from one short aquarium clip of Nity, an octopus, in time order. "
              "Describe in ONE sentence what the octopus is doing.")   # exact training prompt
print("MLX caption student loaded")

## 3. Extraction functions (local-adapted from `extract_octopus_clips.py`)
Same gates as production; the only change is reading a **local file** instead of an authenticated server URL.

In [ ]:
def classify_video_local(path):
    """Per-second p_visible over a local video (letterbox preproc — matches training)."""
    cmd = ["ffmpeg", "-loglevel", "error", "-i", str(path),
           "-vf", (f"fps={SAMPLE_FPS},scale={SIZE}:{SIZE}:force_original_aspect_ratio=decrease,"
                   f"pad={SIZE}:{SIZE}:-1:-1:color=gray"),
           "-f", "image2pipe", "-vcodec", "rawvideo", "-pix_fmt", "rgb24", "-"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    fsize = SIZE * SIZE * 3
    pv, buf = [], []
    def flush():
        if not buf: return
        batch = torch.stack([pre(im) for im in buf]).to(DEV)
        with torch.no_grad():
            f = cm.encode_image(batch).float(); f = f / f.norm(dim=-1, keepdim=True)
            p = torch.softmax(clf(f), dim=1)[:, VIS_IDX]
        pv.extend(p.cpu().tolist()); buf.clear()
    while True:
        raw = proc.stdout.read(fsize)
        if len(raw) < fsize: break
        arr = np.frombuffer(raw, np.uint8).reshape(SIZE, SIZE, 3)
        buf.append(letterbox(Image.fromarray(arr)))
        if len(buf) >= BATCH: flush()
    flush(); proc.stdout.close(); proc.wait()
    return np.array(pv, np.float32)

def motion_local(pv_len, path):
    """Per-second absolute motion aligned to the p_visible index grid."""
    ts, sc = scan_motion_area(str(path), fps=SAMPLE_FPS, pix_thresh=MOTION_PIX)
    m = np.zeros(pv_len, np.float32)
    for k, t in enumerate(ts):
        i = int(round(float(t)))
        if 0 <= i < pv_len: m[i] = sc[k]
    return m

def find_windows(pv, motion):
    """Non-overlapping 20 s windows passing BOTH gates."""
    L = int(CLIP_LEN * SAMPLE_FPS); N = len(pv); out = []; s = 0
    while s + L <= N:
        wp, wm = pv[s:s + L], motion[s:s + L]
        vf = float((wp >= VIS_THRESH).mean()); mm = float(wm.mean())
        if vf > MIN_VISIBLE_FRAC and mm >= MOTION_THRESH:
            out.append({"start": int(s / SAMPLE_FPS), "end": int((s + L) / SAMPLE_FPS),
                        "visible_frac": round(vf, 3), "mean_motion": round(mm, 5)})
            s += L                     # non-overlapping
        else:
            s += 1
    return out

def extract_clip(path, start, end, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and out_path.stat().st_size > 10000: return True
    r = subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-ss", str(start), "-to", str(end),
                        "-i", str(path), "-c:v", "copy", "-c:a", "aac", str(out_path)],
                       capture_output=True, text=True)
    return r.returncode == 0 and out_path.exists()

## 4. Stage 1 — scan the video and extract clips

In [ ]:
t0 = time.time()
print("Stage 1: scanning video (octopus + motion, two 1 fps passes)...", flush=True)
pv = classify_video_local(VIDEO_PATH)
print(f"  octopus pass: {len(pv)} s  (mean p_visible {pv.mean():.2f})  [{time.time()-t0:.0f}s]", flush=True)
motion = motion_local(len(pv), VIDEO_PATH)
print(f"  motion pass : mean {motion.mean():.4f}  max {motion.max():.4f}  [{time.time()-t0:.0f}s]", flush=True)

windows = find_windows(pv, motion)
print(f"  {len(windows)} clip windows pass BOTH gates (>{MIN_VISIBLE_FRAC:.0%} visible & motion>={MOTION_THRESH})")

stem = Path(VIDEO_PATH).stem
clips_dir = OUT_DIR / "clips"
clip_records = []
for w in windows:
    name = f"{CAMERA}_{stem}_{w['start']:04d}-{w['end']:04d}.mp4"
    cp = clips_dir / name
    if extract_clip(VIDEO_PATH, w["start"], w["end"], cp):
        clip_records.append({"clip_path": str(cp), **w,
            "video_timeline": f"{w['start']//60:02d}:{w['start']%60:02d}-{w['end']//60:02d}:{w['end']%60:02d}"})
print(f"  extracted {len(clip_records)} clips -> {clips_dir}  [{time.time()-t0:.0f}s]")

## 5. Stage 2 — caption each clip with the MLX student
Frames are prepared identically to training: dense 1 fps → score → top-6 by p_visible (chronological) →
thumbnail 768 → CLAHE. A clip with no frame ≥ `PRESENT_MIN` is labelled `octopus not present` (skips the VLM).

In [ ]:
def caption_clip(clip_path):
    with tempfile.TemporaryDirectory() as tmp:
        frames = extract_frames(clip_path, tmp)                     # dense 1 fps jpgs
        if not frames:
            return {"caption": None, "status": "noframes"}
        sc = score(frames, cm, pre, clf, VIS_IDX, DEV)              # detector p_visible per frame
        maxp = max(sc)
        if maxp < PRESENT_MIN:                                      # presence gate (skip the VLM)
            return {"caption": "octopus not present", "max_p_visible": round(maxp, 3), "status": "absent"}
        order = sorted(range(len(frames)), key=lambda k: sc[k], reverse=True)[:N_KEEP]
        best  = [frames[k] for k in sorted(order)]                  # keep chronological
        prepped = []
        for j, f in enumerate(best):                               # thumbnail + CLAHE == teacher/training input
            im = Image.open(f).convert("RGB"); im.thumbnail((IMG_MAXSIDE, IMG_MAXSIDE)); im = enhance(im)
            outp = f"{tmp}/best_{j:02d}.jpg"; im.save(outp, quality=90); prepped.append(outp)
        fmt = apply_chat_template(mlx_proc, mlx_model.config, CAP_PROMPT, num_images=len(prepped))
        out = mlx_generate(mlx_model, mlx_proc, fmt, prepped, max_tokens=80, temperature=0.0, verbose=False)
        cap = (out.text if hasattr(out, "text") else out).strip()
        return {"caption": cap, "max_p_visible": round(maxp, 3), "status": "captioned"}

print(f"Stage 2: captioning {len(clip_records)} clips with the local MLX 4-bit student...\n", flush=True)
tc = time.time()
for i, rec in enumerate(clip_records, 1):
    rec.update(caption_clip(rec["clip_path"]))
    print(f"[{i}/{len(clip_records)}] {rec['video_timeline']}  ({rec['status']}, p={rec.get('max_p_visible','-')})")
    print(f"      {rec['caption']}\n", flush=True)
print(f"captioning done in {time.time()-tc:.0f}s")

## 6. Save results

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
result = {
    "video": str(VIDEO_PATH), "camera": CAMERA,
    "processed_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "caption_model": "qwen3vl2b_caption_v1_mlx_4bit",
    "params": {"clip_len": CLIP_LEN, "vis_thresh": VIS_THRESH,
               "min_visible_frac": MIN_VISIBLE_FRAC, "motion_thresh": MOTION_THRESH},
    "n_clips": len(clip_records), "clips": clip_records,
}
out_json = OUT_DIR / f"{Path(VIDEO_PATH).stem}_captions.json"
json.dump(result, open(out_json, "w"), indent=2)

present = [c for c in clip_records if c.get("status") == "captioned"]
print(f"saved -> {out_json}")
print(f"{len(present)}/{len(clip_records)} clips have a present-octopus caption")
print("clips  ->", clips_dir)

## 7. Notes
- **Tuning**: too many junk clips → raise `MOTION_THRESH` (e.g. 0.01) or `VIS_THRESH`; missing real
  events → lower them. `PRESENT_MIN` (in `caption_openrouter`, default 0.5) is the caption presence gate.
- **Speed** (30-min video, Apple Silicon): scan ≈ a few minutes (two 1 fps passes), then ~3 s/clip to caption.
- **View a clip inline**: `from IPython.display import Video; Video(clip_records[0]['clip_path'], embed=True)`.
- This mirrors `extract_octopus_clips.py` (extraction) + the MLX student (captioning); it does **not** write to
  the shared `octopus_clips_verified.json` index — outputs are self-contained under `local_pipeline_out/`.